# **Data Preprocessing**

This notebook is dedicated to cleaning the raw Tanzania Tourism dataset and preparing it for the subsequent feature engineering and machine learning stages.

### Main Objectives of the Preprocessing :

## 1. Import Required Libraries

The following libraries are used throughout the preprocessing stage:

`Path` for handling file and directory paths.

`sys` for managing Python's module search path.

`NumPy` for numerical operations.

`Pandas` for data manipulation and cleaning.

In [1]:
from pathlib import Path
import sys 

notebook_dir = Path.cwd()
project_root = notebook_dir.parent 

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import TRAIN_DATA_PATH, TEST_DATA_PATH, VARIABLE_DEFINITIONS_PATH, NUMERIC_COLS
from src.utils import outliers, massing_analysis
import numpy as np
import pandas as pd



## 2. Load Raw Datasets

load train dataset, test dataset, and variable definitions 

In [2]:
train_df = pd.read_csv(TRAIN_DATA_PATH)
test_df = pd.read_csv(TEST_DATA_PATH)
vb_df = pd.read_csv(VARIABLE_DEFINITIONS_PATH)


In [3]:
train_df.head(5)

,Tour_ID,country,age_group,travel_with,total_female,total_male,purpose,main_activity,info_source,tour_arrangement,...,package_accomodation,package_food,package_transport_tz,package_sightseeing,package_guided_tour,package_insurance,night_mainland,night_zanzibar,first_trip_tz,cost_category
0,tour_id1hffseyw,ITALY,45-64,With Children,0.0,2.0,Visiting Friends and Relatives,Beach Tourism,"Friends, relatives",Package Tour,...,Yes,Yes,Yes,No,No,No,0,7,Yes,High Cost
1,tour_idnacd7zag,UNITED KINGDOM,25-44,With Spouse,1.0,1.0,Leisure and Holidays,Wildlife Tourism,"Travel agent, tour operator",Package Tour,...,Yes,Yes,Yes,No,No,No,0,7,Yes,High Cost
2,tour_id62vz7e71,UNITED STATES OF AMERICA,65+,With Spouse,1.0,1.0,Leisure and Holidays,Widlife Tourism,"Travel agent, tour operator",Package Tour,...,Yes,Yes,Yes,Yes,Yes,No,6,6,Yes,Higher Cost
3,tour_idrc76tzix,RWANDA,25-44,With Spouse and Children,3.0,1.0,Leisure and Holidays,Beach Tourism,"Radio, TV, Web",Independent,...,No,No,No,No,No,No,3,0,No,Lower Cost
4,tour_idn723m0n9,UNITED STATES OF AMERICA,45-64,Alone,0.0,1.0,Leisure and Holidays,Widlife Tourism,"Travel agent, tour operator",Package Tour,...,Yes,Yes,Yes,No,Yes,Yes,7,0,Yes,Higher Cost


In [4]:
train_df.shape


(18506, 21)

In [5]:
test_df.head(5)

,Tour_ID,country,age_group,travel_with,total_female,total_male,purpose,main_activity,info_source,tour_arrangement,package_transport_int,package_accomodation,package_food,package_transport_tz,package_sightseeing,package_guided_tour,package_insurance,night_mainland,night_zanzibar,first_trip_tz
0,tour_idynufedne,KOREA,25-44,Alone,0.0,1.0,Leisure and Holidays,Widlife Tourism,Others,Independent,No,No,No,No,No,No,No,7,4,Yes
1,tour_id9r3y5moe,UNITED KINGDOM,45-64,With Children,1.0,1.0,Leisure and Holidays,Conference Tourism,"Travel agent, tour operator",Package Tour,Yes,Yes,Yes,Yes,Yes,Yes,Yes,7,0,Yes
2,tour_idf6itml6g,ITALY,25-44,With Spouse,1.0,1.0,Leisure and Holidays,Beach Tourism,"Travel agent, tour operator",Package Tour,Yes,Yes,No,Yes,No,No,No,0,6,Yes
3,tour_id99u4znru,KENYA,25-44,Alone,0.0,1.0,Other,Beach Tourism,"Radio, TV, Web",Independent,No,No,No,No,No,No,No,3,4,No
4,tour_idj4i9urbx,ZAMBIA,25-44,Alone,0.0,1.0,Business,Widlife Tourism,"Radio, TV, Web",Independent,No,No,No,No,No,No,No,6,0,No


In [6]:
test_df.shape


(6169, 20)

In [7]:
vb_df

,Column Name,Definition
0,id,Unique identifier for each tourist
1,country,The country a tourist coming from.
2,age_group,The age group of a tourist.
3,travel_with,The relation of people a tourist travel with t...
4,total_female,Total number of females
5,total_male,Total number of males
6,purpose,The purpose of visiting Tanzania
7,main_activity,The main activity of tourism in Tanzania
8,infor_source,The source of information about tourism in Tan...
9,tour_arrangment,The arrangment of visiting Tanzania


## 3. Inspect Data Quality

Review data types, missing values, duplicate records, and outliers before investigating and addressing each issue separately.

**Main points :** 
- Missing Values
- Duplicate Records
- Data Types
- Data Consistency
- Invalid Values

In [8]:
print('train dataframe')
display(train_df.info())
print('_'*50,'\n') 
print('test dataframe')
display(test_df.info())

train dataframe
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18506 entries, 0 to 18505
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Tour_ID                18506 non-null  object 
 1   country                18506 non-null  object 
 2   age_group              18506 non-null  object 
 3   travel_with            17431 non-null  object 
 4   total_female           18504 non-null  float64
 5   total_male             18500 non-null  float64
 6   purpose                18506 non-null  object 
 7   main_activity          18506 non-null  object 
 8   info_source            18506 non-null  object 
 9   tour_arrangement       18506 non-null  object 
 10  package_transport_int  18506 non-null  object 
 11  package_accomodation   18506 non-null  object 
 12  package_food           18506 non-null  object 
 13  package_transport_tz   18506 non-null  object 
 14  package_sightseeing    18506 non-null 

None

__________________________________________________ 

test dataframe
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6169 entries, 0 to 6168
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Tour_ID                6169 non-null   object 
 1   country                6169 non-null   object 
 2   age_group              6169 non-null   object 
 3   travel_with            5808 non-null   object 
 4   total_female           6167 non-null   float64
 5   total_male             6168 non-null   float64
 6   purpose                6169 non-null   object 
 7   main_activity          6169 non-null   object 
 8   info_source            6169 non-null   object 
 9   tour_arrangement       6169 non-null   object 
 10  package_transport_int  6169 non-null   object 
 11  package_accomodation   6169 non-null   object 
 12  package_food           6169 non-null   object 
 13  package_transport_tz   6169 non-null   o

None

In [12]:
print("Missing Values in train dataset : ")
display(massing_analysis(train_df))
print('_'*50,'\n') 
print("Missing Values in test dataset : ")
display(massing_analysis(test_df))


Missing Values in train dataset : 


,number of missing values,percentage of missing values
Tour_ID,,
country,,
age_group,,
travel_with,1075.0,5.81
total_female,2.0,0.01
total_male,6.0,0.03
purpose,,
main_activity,,
info_source,,
tour_arrangement,,


__________________________________________________ 

Missing Values in test dataset : 


,number of missing values,percentage of missing values
Tour_ID,,
country,,
age_group,,
travel_with,361.0,5.85
total_female,2.0,0.03
total_male,1.0,0.02
purpose,,
main_activity,,
info_source,,
tour_arrangement,,


In [13]:
print("Duplicate Records in train dataset : ",train_df.duplicated().sum())
print("Duplicate Records in test dataset : ",test_df.duplicated().sum())


Duplicate Records in train dataset :  0
Duplicate Records in test dataset :  0


In [14]:
print("outliers for train dataset ")
outliers(train_df)
print('_'*50,'\n') 
print("outliers for test dataset ")
outliers(test_df)


outliers for train dataset 
total_female | outliers : 946 - 5.11%
total_male | outliers : 6692 - 36.16%
night_mainland | outliers : 1148 - 6.20%
night_zanzibar | outliers : 729 - 3.94%
__________________________________________________ 

outliers for test dataset 
total_female | outliers : 320 - 5.19%
total_male | outliers : 2204 - 35.73%
night_mainland | outliers : 383 - 6.21%
night_zanzibar | outliers : 271 - 4.39%
